<a href="https://colab.research.google.com/github/waheddevs/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print('Menu Detector')

Menu Detector


In [13]:
# ----------------
# Import libraries
# ----------------
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms as transform
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np


In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# Define dataset path
DATASET_PATH = '/content/drive/MyDrive/food101'


# Label grouping | class consolidation
CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert',
    'cheesecake': 'dessert',
    'kebab': 'kebab',
    'plov': 'plov'
}


CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

transform.Compose([         # Compose - used for orders of situations. Its like a pipelines
    transform.Resize((224, 224)),
    transform.ToTensor(),
    transform.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# -----------------------
# Custom Dataset Class
# -----------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('image_path', img_path)
        label = self.labels[idx]
        print('label', label)
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))
        if self.transform:
            image = self.transform(image)
        return image, label

5
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [8]:
# --------------------------
# Gather and Split Data
# --------------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print('class_path:', class_path)
    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue
    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101/hamburger
class_path: /content/drive/MyDrive/food101/hot_dog
class_path: /content/drive/MyDrive/food101/chocolate_cake
class_path: /content/drive/MyDrive/food101/cheesecake
class_path: /content/drive/MyDrive/food101/kebab
class_path: /content/drive/MyDrive/food101/plov
images_length 3240
3240
image_path /content/drive/MyDrive/food101/cheesecake/3824062.jpg
label 2


In [9]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)


In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)  # thread (parallel loading for speed)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

images_length 3240
images_length 3240


In [11]:
# pretrained model
model = mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)   # fine-tuning

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 114MB/s]


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

device cuda


In [16]:
criterion = nn.CrossEntropyLoss()   # Loss Function for prediction('70%' burger, '30%' plov)
optimizer = optim.Adam(model.parameters(), lr=0.001)   # updating parametres => weights
torch.backends.cudnn.benchmark = True   # Benchmark Settings => Trick (for available algorithm)

print('criterion:', criterion)
print('optimizer:', optimizer)

criterion: CrossEntropyLoss()
optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
